# Research Question 1

**Can downprojection reveal systematic patterns in the model’s false-positive and false-negative predictions?**

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.neighbors import NearestNeighbors
from IPython.utils.capture import capture_output

In [2]:
# Import data from the base notebook

# Use "capture" do suppress plotting images
with capture_output() as captured:
    %run r0_dataset_overiew_dashboard.ipynb

## 0 Configs

In [3]:
HTML_OUTPUT_FOLDER = "html_plots/r1"

## 1 Show and compare downprojections

In [4]:
print_dataset_stats()

Dataset statistics
Total number of samples: 10000
Number of pathologies: 11

Atelectasis               positive:  1915 | negative:  8085 | uncertain:     0
Cardiomegaly              positive:  1494 | negative:  8506 | uncertain:     0
Consolidation             positive:   857 | negative:  9143 | uncertain:     0
Edema                     positive:  2240 | negative:  7760 | uncertain:     0
Enlarged Cardiomediastinum positive:   833 | negative:  9167 | uncertain:     0
Fracture                  positive:   672 | negative:  9328 | uncertain:     0
Lung Lesion               positive:   670 | negative:  9330 | uncertain:     0
Lung Opacity              positive:  4806 | negative:  5194 | uncertain:     0
Effusion                  positive:  3214 | negative:  6786 | uncertain:     0
Pneumonia                 positive:   439 | negative:  9561 | uncertain:     0
Pneumothorax              positive:   679 | negative:  9321 | uncertain:     0


In [5]:
visualization_functions = [
    visualize_space_traces,
    visualize_space_prediction_scores_traces,
    visualize_classification_outcome_traces
]

visualization_names = [
    "Groundtruth with Gender",
    "Model Confidence Score",
    "Model Classification outcomes",
]
# NOTE: This creates heavy plot, pls use the plotHTML flag.
create_subplot_for_visualization_fn(
        downprojector=DOWN_PROJECTOR,
        projection_method="S-UMAP",
        visualization_fns=visualization_functions,
        visualization_names= visualization_names,
        model_score_predictions=MODEL_SCORE_PREDICTIONS_CONVERTED,
        ground_truth=XRAY_LABELS_LIST,
        dataset_label_indices_map=DATASET_LABEL_INDICES_MAP,
        dataset_gender_list=XRAY_GENDER_LIST,
        dataset_ages=XRAY_AGE_LIST,
        threshold=THRESHOLD,
        plotHTML=True,
        output_file='html_plots/r1/downprojection_subplot.html'
)

['Atelectasis - Groundtruth with Gender', 'Atelectasis - Model Confidence Score', 'Atelectasis - Model Classification outcomes', 'Cardiomegaly - Groundtruth with Gender', 'Cardiomegaly - Model Confidence Score', 'Cardiomegaly - Model Classification outcomes', 'Consolidation - Groundtruth with Gender', 'Consolidation - Model Confidence Score', 'Consolidation - Model Classification outcomes', 'Edema - Groundtruth with Gender', 'Edema - Model Confidence Score', 'Edema - Model Classification outcomes', 'Enlarged Cardiomediastinum - Groundtruth with Gender', 'Enlarged Cardiomediastinum - Model Confidence Score', 'Enlarged Cardiomediastinum - Model Classification outcomes', 'Fracture - Groundtruth with Gender', 'Fracture - Model Confidence Score', 'Fracture - Model Classification outcomes', 'Lung Lesion - Groundtruth with Gender', 'Lung Lesion - Model Confidence Score', 'Lung Lesion - Model Classification outcomes', 'Lung Opacity - Groundtruth with Gender', 'Lung Opacity - Model Confidence S

### Observation

After the downprojection and visualization of the pathology labels, two distinct types of clustering patterns can be observed.

The first type consists of pathologies where the positive and negative samples form two clearly distinguishable clusters. This pattern can be observed for Atelectasis, Cardiomegaly, Edema, Lung Opacity, and Effusion. For these pathologies, the model generally performed relatively well compared with the other findings, particularly for Edema, Effusion, Lung Opacity, and Cardiomegaly. The clear separation between positive and negative samples in the projection appears to correspond to the model's comparatively stronger ability to distinguish between the two classes.

The second type consists of a large manifold surrounded by smaller clusters or individual samples. In most cases, the large central manifold primarily contains the negative cases, while the smaller satellite clusters consist mainly of positive cases. This pattern can be observed for Pneumonia, Consolidation, Lung Lesion, Pneumothorax, and Fracture. For these pathologies, the model performed considerably worse. In some cases, it was barely able to detect any positive cases, suggesting that the positive samples are less clearly separated from the dominant negative manifold in the learned representation.

One possible reason could be that for the pathologies of the second cluster type there are less samples.

Note: This observation appeared for both CheX and Google NIH datasets.

## 2 Show local error rate in downprojected space

As suggested by the AI: For every point in the projection, examine its nearest neighbors and calculate their local false-positive and false-negative rates.

In [6]:
def create_local_error_maps(
    projection_method: str,
    down_projector: Downprojector,
    ground_truth,
    model_scores,
    pathology,
    label_indices_map,
    threshold=THRESHOLD,
    k=100,
    min_class_count=20,
    target_labels=XRAY_LABELS_LIST,
    showPlotInNotebook=False,
    output_file=None
):

    label_idx = label_indices_map[pathology]

    y_true = ground_truth[:, label_idx]
    y_scores = model_scores[:, label_idx]

    y_pred = (y_scores >= threshold).astype(int)

    embedding = down_projector.calculate_projection(
        "feature",
        projection_method,
        target_labels=target_labels,
        label_name=pathology
    )

    coordinates = np.asarray(embedding, dtype=float)

    positives = y_true == 1
    negatives = y_true == 0

    global_fn = np.sum(positives & (y_pred == 0))
    global_tp = np.sum(positives & (y_pred == 1))

    global_fp = np.sum(negatives & (y_pred == 1))
    global_tn = np.sum(negatives & (y_pred == 0))

    global_fnr = global_fn / (global_fn + global_tp)
    global_fpr = global_fp / (global_fp + global_tn)

    # Find k nearest neighbors for every point
    k = min(k, len(coordinates))

    nn = NearestNeighbors(n_neighbors=k)
    nn.fit(coordinates)

    neighbor_indices = nn.kneighbors(
        coordinates,
        return_distance=False
    )

    # local FNR and FPR
    local_fnr = np.full(len(coordinates), np.nan)
    local_fpr = np.full(len(coordinates), np.nan)

    fn_counts = np.zeros(len(coordinates), dtype=int)
    positive_counts = np.zeros(len(coordinates), dtype=int)

    fp_counts = np.zeros(len(coordinates), dtype=int)
    negative_counts = np.zeros(len(coordinates), dtype=int)

    for i, neighbors in enumerate(neighbor_indices):


        neighbor_true = y_true[neighbors]
        neighbor_pred = y_pred[neighbors]

        # Positive ground-truth cases
        pos = neighbor_true == 1

        n_pos = np.sum(pos)
        n_fn = np.sum(pos & (neighbor_pred == 0))

        positive_counts[i] = n_pos
        fn_counts[i] = n_fn

        if n_pos >= min_class_count:
            local_fnr[i] = n_fn / n_pos

        # Negative ground-truth cases
        neg = neighbor_true == 0

        n_neg = np.sum(neg)
        n_fp = np.sum(neg & (neighbor_pred == 1))

        negative_counts[i] = n_neg
        fp_counts[i] = n_fp

        if n_neg >= min_class_count:
            local_fpr[i] = n_fp / n_neg

    def plot_error_map(rate, error_counts, denominators,
                       global_rate, title):

        mask = np.isfinite(rate)

        customdata = np.column_stack([
            error_counts[mask],
            denominators[mask]
        ])

        fig = go.Figure()

        fig.add_trace(go.Scatter(
            x=coordinates[mask, 0],
            y=coordinates[mask, 1],
            mode="markers",

            marker=dict(
                size=6,
                color=rate[mask],
                colorscale="RdYlBu_r",
                cmin=0,
                cmax=1,
                colorbar=dict(title="Local error rate"),
                opacity=0.85
            ),

            customdata=customdata,

            hovertemplate=(
                "Local error rate: %{marker.color:.1%}<br>"
                "Errors: %{customdata[0]}<br>"
                "Relevant cases: %{customdata[1]}<br>"
                "<extra></extra>"
            )
        ))

        fig.update_layout(
            title=(
                f"{pathology} - {title}<br>"
                f"<sup>Global rate: {global_rate:.1%} | "
                f"k={k} | Minimum denominator={min_class_count}</sup>"
            ),
            xaxis_title=f"{projection_method} 1",
            yaxis_title=f"{projection_method} 2",
            template="plotly_white",
            width=900,
            height=700
        )

        return fig
    
    fnr_fig = plot_error_map(
        local_fnr,
        fn_counts,
        positive_counts,
        global_fnr,
        "Local False Negative Rate"
    )

    fpr_fig = plot_error_map(
        local_fpr,
        fp_counts,
        negative_counts,
        global_fpr,
        "Local False Positive Rate"
    )

    print(f"Pathology: {pathology}")
    print(f"Threshold: {threshold}")
    print(f"Global FNR: {global_fnr:.2%}")
    print(f"Global FPR: {global_fpr:.2%}")
    if output_file is not None:
        fnr_fig.write_html(
            output_file,
            include_plotlyjs="cdn",
            full_html=True
        )
        fpr_fig.write_html(
            output_file,
            include_plotlyjs="cdn",
            full_html=True
        )
    if showPlotInNotebook:
        fnr_fig.show()
        fpr_fig.show()

    return {
        "local_fnr": local_fnr,
        "local_fpr": local_fpr,
        "positive_counts": positive_counts,
        "negative_counts": negative_counts,
        "fn_counts": fn_counts,
        "fp_counts": fp_counts,
        "global_fnr": global_fnr,
        "global_fpr": global_fpr
    }


In [7]:
# plot for every pathology
for pathology in XRAY_DATASET.pathologies:


    results = create_local_error_maps(
        projection_method='PCA',
        down_projector=DOWN_PROJECTOR,
        ground_truth=XRAY_LABELS_LIST,
        model_scores=MODEL_SCORE_PREDICTIONS_CONVERTED,
        pathology=pathology,
        label_indices_map=DATASET_LABEL_INDICES_MAP,
        threshold=THRESHOLD,
        k=100,
        min_class_count=20,
        showPlotInNotebook=True,
        output_file=HTML_OUTPUT_FOLDER + f'/{pathology}_local_error.html'
    )


Pathology: Atelectasis
Threshold: 0.7
Global FNR: 29.14%
Global FPR: 48.89%


Pathology: Cardiomegaly
Threshold: 0.7
Global FNR: 24.77%
Global FPR: 31.52%


Pathology: Consolidation
Threshold: 0.7
Global FNR: 41.77%
Global FPR: 25.40%


Pathology: Edema
Threshold: 0.7
Global FNR: 25.71%
Global FPR: 32.56%


Pathology: Enlarged Cardiomediastinum
Threshold: 0.7
Global FNR: 65.91%
Global FPR: 19.77%


Pathology: Fracture
Threshold: 0.7
Global FNR: 82.14%
Global FPR: 10.77%


Pathology: Lung Lesion
Threshold: 0.7
Global FNR: 72.09%
Global FPR: 13.04%


Pathology: Lung Opacity
Threshold: 0.7
Global FNR: 23.72%
Global FPR: 38.89%


Pathology: Effusion
Threshold: 0.7
Global FNR: 19.17%
Global FPR: 29.86%


Pathology: Pneumonia
Threshold: 0.7
Global FNR: 70.84%
Global FPR: 14.47%


Pathology: Pneumothorax
Threshold: 0.7
Global FNR: 98.09%
Global FPR: 0.23%
